---
# PHẦN 1 — PGA-UNet (IMG_SIZE=256)
---

# PGA-UNet — Test & Visualization (IMG_SIZE=256)
**Checkpoint**: điền CKPT_ID vào cell Setup

Chạy tuần tự: **Setup → Test → Visualization**

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
%cd /kaggle/working
import os, gdown, torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Clone repo
if not os.path.exists('Prompt-Guided-XRay-Segmentation'):
    !git clone -b TN_B_ON https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git
else:
    !cd Prompt-Guided-XRay-Segmentation && git pull -q

# Dataset
!gdown --id 1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW -O /kaggle/working/dataset_BTXRD.zip -q
!unzip -oq /kaggle/working/dataset_BTXRD.zip -d /kaggle/working/
!rsync -a /kaggle/working/dataset_BTXRD/ /kaggle/working/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/ 2>/dev/null

%cd /kaggle/working/Prompt-Guided-XRay-Segmentation
!pip install -q tqdm opencv-python matplotlib gdown scipy

# ── Checkpoint ─────────────────────────────────────────────────────
CKPT_ID   = '1GUgKTTOtAs7thFc7R1TPRTKc0dVSlW_e'          # ← điền ID Google Drive, ví dụ: '1abc...'
CKPT_PATH = '/kaggle/working/Prompt-Guided-XRay-Segmentation/checkpoints/pga_unet_expB_best.pth'
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
assert CKPT_ID, '❌ Chưa điền CKPT_ID — upload checkpoint lên Drive rồi điền ID vào đây'
gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_PATH, quiet=False)
assert os.path.exists(CKPT_PATH)
print(f'\n✅ Setup xong  |  {os.path.getsize(CKPT_PATH)//1024} KB')


## Test — 3 prompt modes (image-level merging)

In [ ]:
# ── Test: 3 prompt modes — image-level merging ─────────────────────
import os, cv2, csv
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict
import sys

# ── Đảm bảo import đúng PGA dataset (tránh cache từ SAM-Med2D hoặc unet) ──────
_PGA_PATH = '/kaggle/working/Prompt-Guided-XRay-Segmentation'
for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset', 'models', 'prompt_unet', 'unet')):
        del sys.modules[_k]
if _PGA_PATH in sys.path: sys.path.remove(_PGA_PATH)
sys.path.insert(0, _PGA_PATH)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

# Kiểm tra nhanh signature để chắc đúng file
import inspect as _ins
assert 'prompt_mode' in _ins.signature(BTXRD_Dataset.__init__).parameters, \
    f'❌ Import sai dataset.py! Thực tế từ: {BTXRD_Dataset.__module__}'
print(f'✅ BTXRD_Dataset OK — {_PGA_PATH}/dataset.py')

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 256
IMG_DIR  = f'{_PGA_PATH}/dataset_BTXRD/test/images'
JSON_DIR = f'{_PGA_PATH}/dataset_BTXRD/test/annotations'
os.makedirs(f'{_PGA_PATH}/results', exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ Model loaded  device={DEVICE}')

def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95), np.percentile(d2,95)))

def calc_metrics_img(prob_np, gt_np):
    pm = (prob_np > 0.5).astype(np.float32)
    gm = (gt_np  > 0.5).astype(np.float32); eps = 1e-6
    tp = (pm * gm).sum(); fp = (pm * (1-gm)).sum(); fn = ((1-pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        bbox_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2) / bbox_diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS  = ['dice', 'iou', 'precision', 'recall', 'hd95', 'cbl']
HDRS  = ['Dice\u2191', 'IoU\u2191', 'Prec\u2191', 'Rec\u2191', 'HD95\u2193', 'CBL\u2191']
MODES = ['zoom_out', 'shift', 'mixed_7_3']
all_image_records = {}
pga_results = {}   # dùng ở cell tổng hợp
csv_rows = []

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR, JSON_DIR, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)

    groups = OrderedDict()
    for i, (img_t, mask_t, prompt_t) in enumerate(tqdm(loader, desc=f'[{mode}]')):
        img_name, _ = ds.all_samples[i]
        gt_np     = mask_t[0, 0].numpy()
        prompt_np = prompt_t[0, 0].numpy()
        with torch.no_grad():
            prob = torch.sigmoid(model(img_t.to(DEVICE), prompt_t.to(DEVICE)))[0, 0].cpu().numpy()
        if img_name not in groups:
            groups[img_name] = dict(img=img_t[0, 0].numpy(),
                                    gt_union=gt_np.copy(),
                                    prob_max=prob.copy(),
                                    prompts=[prompt_np])
        else:
            g = groups[img_name]
            np.maximum(g['gt_union'], gt_np, out=g['gt_union'])
            np.maximum(g['prob_max'], prob,  out=g['prob_max'])
            g['prompts'].append(prompt_np)

    img_recs = []
    for img_name in sorted(groups.keys()):
        g = groups[img_name]
        m = calc_metrics_img(g['prob_max'], g['gt_union'])
        img_recs.append(dict(img_name=img_name,
                             img=g['img'], gt=g['gt_union'], prob=g['prob_max'],
                             prompts=g['prompts'], n_samples=len(g['prompts']), **m))

    all_image_records[mode] = img_recs
    m_avg  = {k: np.mean([r[k] for r in img_recs]) for k in KEYS}
    pga_results[mode] = m_avg
    n_imgs = len(img_recs)
    n_samp = sum(r['n_samples'] for r in img_recs)
    csv_rows.append([mode] + [f'{m_avg[k]:.4f}' for k in KEYS] + [str(n_imgs), str(n_samp)])

bar = '=' * 82
print(f'\n{bar}\n  PGA-UNet — Image-level metrics (GT union + max-merge)  IMG_SIZE={IMG_SIZE}\n{bar}')
print(f"  {'Mode':<16}" + ''.join(f'{h:>8}' for h in HDRS) + f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in csv_rows:
    print(f"  {row[0]:<16}" + ''.join(f'{row[i+1]:>8}' for i in range(len(KEYS))) + f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open(f'{_PGA_PATH}/results/pga_unet2d_r256_test_results.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['mode'] + KEYS + ['N_img', 'N_samples'])
    w.writerows(csv_rows)
print(f'\n✅ CSV: {_PGA_PATH}/results/pga_unet2d_r256_test_results.csv')


## Visualization — ảnh có ≥2 GT polygon

In [ ]:
# ── Visualization: ≥10 ảnh có từ 2 GT polygon trở lên (zoom_out) ───
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Chạy cell Test trước'

N_MIN  = 10
MIN_GT = 2

recs = [r for r in all_image_records['zoom_out'] if r['n_samples'] >= MIN_GT]
assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs = recs[:max(N_MIN, len(recs))]
N_SHOW = len(recs)

fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'PGA-UNet — {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

for count, rec in enumerate(recs):
    img_np  = rec['img'] * 0.5 + 0.5
    gt_np   = (rec['gt']   > 0.5).astype(float)
    pred_np = (rec['prob'] > 0.5).astype(float)
    prompt_merged = np.max(np.stack(rec['prompts'], axis=0), axis=0)

    tp = (pred_np * gt_np).sum(); fp = (pred_np * (1-gt_np)).sum()
    fn = ((1-pred_np) * gt_np).sum(); e = 1e-6
    dice = float((2*tp+e)/(2*tp+fp+fn+e)); iou  = float((tp+e)/(tp+fp+fn+e))
    pre  = float((tp+e)/(tp+fp+e));         rec_ = float((tp+e)/(tp+fn+e))
    n    = rec['n_samples']

    row = axes[count]
    bg  = np.stack([img_np]*3, axis=-1)

    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[1].imshow(prompt_merged, cmap='hot', alpha=0.4, vmin=0, vmax=1)
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np       * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1-gt_np)   * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1-pred_np) * gt_np   * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)


---
# PHẦN 2 — SAM-Med2D Fine-tuned
---

# SAM-Med2D Fine-tuned — Test Only (Robust Bbox)

Thứ tự chạy: **Setup → Chuẩn bị → (%%writefile) → Test A / B / C → Kết quả**

In [ ]:
%cd /kaggle/working

# ── Cài đặt gói ──────────────────────────────────────────────────
!pip install -q albumentations gdown

import os, gdown, cv2, json as _json, numpy as np, glob

# ── Clone SAM-Med2D ───────────────────────────────────────────────
if not os.path.exists('SAM-Med2D'):
    !git clone -q https://github.com/OpenGVLab/SAM-Med2D/

%cd /kaggle/working/SAM-Med2D

# ── Dataset ───────────────────────────────────────────────────────
if not os.path.exists('dataset_BTXRD'):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   '/tmp/dataset_BTXRD.zip', quiet=False)
    !unzip -oq /tmp/dataset_BTXRD.zip -d /kaggle/working/SAM-Med2D/

# ── Per-polygon masks: JSON annotation → PNG riêng từng polygon ──
def build_per_polygon_masks(split):
    ann_dir  = f"dataset_BTXRD/{split}/annotations"
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"
    if not os.path.exists(ann_dir):
        print(f"[!] {split}: không có annotations, bỏ qua"); return
    for p in glob.glob(os.path.join(mask_dir, "*.png")): os.remove(p)
    n = 0
    for jf in sorted(glob.glob(os.path.join(ann_dir, "*.json"))):
        base = os.path.splitext(os.path.basename(jf))[0]
        img_path = next(
            (os.path.join(img_dir, base + e) for e in ('.png', '.jpg', '.jpeg')
             if os.path.exists(os.path.join(img_dir, base + e))), None)
        if img_path is None: continue
        img  = cv2.imread(img_path)
        H, W = img.shape[:2]
        data = _json.load(open(jf, encoding='utf-8'))
        polys = [s for s in data.get('shapes', []) if s.get('shape_type') == 'polygon']
        for i, shp in enumerate(polys, 1):
            pts  = np.array(shp['points'], dtype=np.float32).reshape(-1, 1, 2).astype(np.int32)
            mask = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            cv2.imwrite(os.path.join(mask_dir, f"{base}_{i}.png"), mask)
            n += 1
    print(f"[✅] {split}: {n} per-polygon masks")

for split in ['train', 'val', 'test']:
    build_per_polygon_masks(split)

# ── Fine-tuned checkpoint ─────────────────────────────────────────
CKPT_ID    = '1hGK6_4WCmH4Tzd2XJcerQFYMeDLSiYsg'
CKPT_LOCAL = '/kaggle/working/SAM-Med2D/checkpoint/best_sam.pth'
os.makedirs(os.path.dirname(CKPT_LOCAL), exist_ok=True)
if not os.path.exists(CKPT_LOCAL):
    gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_LOCAL, quiet=False)
assert os.path.exists(CKPT_LOCAL), '❌ Tải checkpoint thất bại'
print(f'\n✅ Setup xong | {CKPT_LOCAL}  ({os.path.getsize(CKPT_LOCAL)//1024//1024} MB)')

## Phần 2 – Thí nghiệm Đánh giá

Ba kịch bản prompt bbox:
- **A – Zoom-Out ~30%**: bbox mở rộng đều 4 phía 30%
- **B – Shift ~30%**: bbox lệch vị trí ~30%
- **C – Mixed 70/30**: 70% zoom-out + 30% shift

### 2.0 – Chuẩn bị (chạy 1 lần trước khi test)

In [ ]:
%cd /kaggle/working/SAM-Med2D
import os, json, glob, gdown

# ── Tạo label2image_test.json & label2image_val.json ─────────────
def create_evaluation_json(split="test"):
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"
    label2image = {}
    if not os.path.exists(mask_dir) or not os.path.exists(img_dir):
        print(f"[!] Không tìm thấy thư mục cho tập: '{split}'"); return
    img_dict = {}
    for f in os.listdir(img_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_dict[os.path.splitext(f)[0]] = f
    for mask_name in sorted(os.listdir(mask_dir)):
        if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        mask_base = os.path.splitext(mask_name)[0]
        img_base  = mask_base.rsplit('_', 1)[0] if '_' in mask_base else mask_base
        if img_base in img_dict:
            label2image[f"dataset_BTXRD/{split}/masks/{mask_name}"] = \
                        f"dataset_BTXRD/{split}/images/{img_dict[img_base]}"
    out = f"dataset_BTXRD/label2image_{split}.json"
    with open(out, "w", encoding="utf-8") as fo:
        json.dump(label2image, fo, indent=4, ensure_ascii=False)
    print(f"[*] label2image_{split}.json — {len(label2image)} mẫu → {out}")

create_evaluation_json("test")
create_evaluation_json("val")

# ── Load CHECKPOINT (Drive-mount → checkpoint/ → workdir → tải về) ───────────
CKPT_DRIVE_ID = '1hGK6_4WCmH4Tzd2XJcerQFYMeDLSiYsg'
CKPT_LOCAL    = '/kaggle/working/SAM-Med2D/checkpoint/best_sam.pth'

def find_ckpt(folder):
    for name in ["best_sam.pth", "last_sam.pth"]:
        p = os.path.join(folder, name)
        if os.path.exists(p): return p
    return None

FT_CHECKPOINT = (find_ckpt("/kaggle/working/drive/MyDrive/model") or
              find_ckpt("/kaggle/working/SAM-Med2D/checkpoint") or
              find_ckpt("/kaggle/working/SAM-Med2D/workdir/models/sam-med2d"))

if FT_CHECKPOINT:
    src = "Drive-mount" if "MyDrive" in FT_CHECKPOINT else "Local"
    print(f"[{src}] ✅ {FT_CHECKPOINT}")
else:
    print("[Download] Tải fine-tuned checkpoint từ Google Drive...")
    os.makedirs(os.path.dirname(CKPT_LOCAL), exist_ok=True)
    gdown.download(f'https://drive.google.com/uc?id={CKPT_DRIVE_ID}', CKPT_LOCAL, quiet=False)
    FT_CHECKPOINT = CKPT_LOCAL

assert os.path.exists(FT_CHECKPOINT), "❌ Checkpoint không tồn tại!"
print(f"\n✅ Checkpoint: {FT_CHECKPOINT}  ({os.path.getsize(FT_CHECKPOINT)//1024//1024} MB)")

In [ ]:
%%writefile /kaggle/working/SAM-Med2D/segment_anything/build_sam.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.

import torch
from functools import partial
from .modeling import ImageEncoderViT, MaskDecoder, PromptEncoder, Sam, TwoWayTransformer
from torch.nn import functional as F

def build_sam_vit_h(args):
    return _build_sam(
        encoder_embed_dim=1280,
        encoder_depth=32,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[7, 15, 23, 31],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter=args.encoder_adapter,
    )

build_sam = build_sam_vit_h

def build_sam_vit_l(args):
    return _build_sam(
        encoder_embed_dim=1024,
        encoder_depth=24,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[5, 11, 17, 23],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter=args.encoder_adapter,
    )

def build_sam_vit_b(args):
    return _build_sam(
        encoder_embed_dim=768,
        encoder_depth=12,
        encoder_num_heads=12,
        encoder_global_attn_indexes=[2, 5, 8, 11],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter=args.encoder_adapter,
    )

sam_model_registry = {
    "default": build_sam_vit_h,
    "vit_h":   build_sam_vit_h,
    "vit_l":   build_sam_vit_l,
    "vit_b":   build_sam_vit_b,
}

def _build_sam(encoder_embed_dim, encoder_depth, encoder_num_heads,
               encoder_global_attn_indexes, image_size, checkpoint, encoder_adapter):
    prompt_embed_dim = 256
    vit_patch_size   = 16
    image_embedding_size = image_size // vit_patch_size
    sam = Sam(
        image_encoder=ImageEncoderViT(
            depth=encoder_depth,
            embed_dim=encoder_embed_dim,
            img_size=image_size,
            mlp_ratio=4,
            norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
            num_heads=encoder_num_heads,
            patch_size=vit_patch_size,
            qkv_bias=True,
            use_rel_pos=True,
            global_attn_indexes=encoder_global_attn_indexes,
            window_size=14,
            out_chans=prompt_embed_dim,
            adapter_train=encoder_adapter,
        ),
        prompt_encoder=PromptEncoder(
            embed_dim=prompt_embed_dim,
            image_embedding_size=(image_embedding_size, image_embedding_size),
            input_image_size=(image_size, image_size),
            mask_in_chans=16,
        ),
        mask_decoder=MaskDecoder(
            num_multimask_outputs=3,
            transformer=TwoWayTransformer(
                depth=2,
                embedding_dim=prompt_embed_dim,
                mlp_dim=2048,
                num_heads=8,
            ),
            transformer_dim=prompt_embed_dim,
            iou_head_depth=3,
            iou_head_hidden_dim=256,
        ),
        pixel_mean=[123.675, 116.28, 103.53],
        pixel_std=[58.395, 57.12, 57.375],
    )
    if checkpoint is not None:
        with open(checkpoint, "rb") as f:
            state_dict = torch.load(f, map_location="cpu", weights_only=False)
        try:
            if 'model' in state_dict:
                print(encoder_adapter)
                sam.load_state_dict(state_dict['model'], strict=False)
            else:
                if image_size == 1024 and encoder_adapter:
                    sam.load_state_dict(state_dict, strict=False)
                else:
                    sam.load_state_dict(state_dict)
        except Exception:
            print('**** interpolate pos embed')
            new_state_dict = load_from(sam, state_dict, image_size, vit_patch_size)
            sam.load_state_dict(new_state_dict)
        print(f"**** loaded {checkpoint}")
    return sam

def load_from(sam, state_dicts, image_size, vit_patch_size):
    sam_dict    = sam.state_dict()
    skip_keys   = ['mask_tokens', 'output_hypernetworks_mlps', 'iou_prediction_head']
    new_sd      = {k: v for k, v in state_dicts.items()
                   if k in sam_dict and not any(s in k for s in skip_keys)}
    pos_embed   = new_sd.get('image_encoder.pos_embed')
    token_size  = image_size // vit_patch_size
    if pos_embed is not None and pos_embed.shape[1] != token_size:
        pos_embed = F.interpolate(
            pos_embed.permute(0,3,1,2), (token_size, token_size),
            mode='bilinear', align_corners=False).permute(0,2,3,1)
        new_sd['image_encoder.pos_embed'] = pos_embed
        for k in [k for k in sam_dict if 'rel_pos' in k and
                  any(t in k for t in ['2','5','7','8','11','13','15','23','31'])]:
            rp = new_sd.get(k)
            if rp is not None:
                h_c, w_c = sam_dict[k].shape
                if rp.shape != (h_c, w_c):
                    rp = F.interpolate(rp.unsqueeze(0).unsqueeze(0),
                                       (h_c, w_c), mode='bilinear', align_corners=False)[0,0]
                new_sd[k] = rp
    sam_dict.update(new_sd)
    return sam_dict

In [ ]:
%%writefile /kaggle/working/SAM-Med2D/DataLoader.py
import os, json, random
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
from torch.nn import functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from utils import train_transforms, get_boxes_from_mask, init_point_sampling


class TestingDataset(Dataset):
    def __init__(self, data_path, image_size=256, mode='test', requires_name=True,
                 point_num=1, return_ori_mask=True, prompt_path=None,
                 prompt_mode='zoom_out', zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        self.image_size    = image_size
        self.return_ori_mask = return_ori_mask
        self.prompt_path   = prompt_path
        self.prompt_list   = {} if prompt_path is None else json.load(open(prompt_path))
        self.requires_name = requires_name
        self.point_num     = point_num
        self.mode          = mode
        self.is_train      = (mode == 'train')
        self.prompt_mode   = prompt_mode
        self.zoom_ratio    = zoom_ratio
        self.shift_ratio   = shift_ratio

        dataset = json.load(open(os.path.join(data_path, f'label2image_{mode}.json')))
        sorted_items       = sorted(dataset.items(), key=lambda x: os.path.basename(x[0]))
        self.label_paths   = [k for k, v in sorted_items]
        self.image_paths   = [v for k, v in sorted_items]
        self.pixel_mean    = [123.675, 116.28, 103.53]
        self.pixel_std     = [58.395, 57.12, 57.375]

    def _zoom_out_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        lo, hi = self.zoom_ratio
        if self.is_train:
            r_l, r_r = random.uniform(lo, hi), random.uniform(lo, hi)
            r_t, r_b = random.uniform(lo, hi), random.uniform(lo, hi)
        else:
            r = (lo + hi) / 2
            r_l = r_r = r_t = r_b = r
        return (max(0, x_min - gt_w*r_l), min(orig_w, x_max + gt_w*r_r),
                max(0, y_min - gt_h*r_t), min(orig_h, y_max + gt_h*r_b))

    def _shift_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=None):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(
            x_min, x_max, y_min, y_max, orig_h, orig_w)
        if self.is_train:
            dx = random.uniform(-gt_w * self.shift_ratio, gt_w * self.shift_ratio)
            dy = random.uniform(-gt_h * self.shift_ratio, gt_h * self.shift_ratio)
        else:
            rng = random.Random(seed_idx or 0)
            dx  = rng.uniform(gt_w * 0.4, gt_w * 0.7) * self.shift_ratio
            dy  = rng.uniform(gt_h * 0.1, gt_h * 0.3) * self.shift_ratio
        bx_min = max(0,      bx_min + dx);  bx_max = min(orig_w, bx_max + dx)
        by_min = max(0,      by_min + dy);  by_max = min(orig_h, by_max + dy)
        if min(bx_max, x_max) - max(bx_min, x_min) < gt_w * 0.3:
            if dx > 0: bx_max = min(orig_w, x_max + gt_w * 0.15)
            else:      bx_min = max(0,      x_min - gt_w * 0.15)
        if min(by_max, y_max) - max(by_min, y_min) < gt_h * 0.3:
            if dy > 0: by_max = min(orig_h, y_max + gt_h * 0.15)
            else:      by_min = max(0,      y_min - gt_h * 0.15)
        return bx_min, bx_max, by_min, by_max

    def __getitem__(self, index):
        image = cv2.imread(self.image_paths[index])
        image = (image - self.pixel_mean) / self.pixel_std

        mask_path  = self.label_paths[index]
        ori_np_mask = cv2.imread(mask_path, 0)
        if ori_np_mask.max() == 255:
            ori_np_mask = ori_np_mask / 255
        assert np.array_equal(ori_np_mask, ori_np_mask.astype(bool)), \
            f"Mask phải là binary. {mask_path}"

        h, w = ori_np_mask.shape
        ori_mask = torch.tensor(ori_np_mask).unsqueeze(0)

        transforms = train_transforms(self.image_size, h, w)
        aug   = transforms(image=image, mask=ori_np_mask)
        image, mask = aug['image'], aug['mask'].to(torch.int64)

        if self.prompt_path is None:
            y_idx, x_idx = torch.where(mask > 0)
            if len(y_idx) > 0:
                y_min, y_max = y_idx.min().item(), y_idx.max().item()
                x_min, x_max = x_idx.min().item(), x_idx.max().item()
            else:
                x_min, x_max, y_min, y_max = 0, self.image_size, 0, self.image_size
            S = self.image_size
            if self.prompt_mode == 'zoom_out':
                bx0, bx1, by0, by1 = self._zoom_out_bbox(x_min, x_max, y_min, y_max, S, S)
            elif self.prompt_mode == 'shift':
                bx0, bx1, by0, by1 = self._shift_bbox(x_min, x_max, y_min, y_max, S, S, index)
            elif self.prompt_mode == 'mixed_7_3':
                use_shift = (random.random() < 0.3) if self.is_train else (index % 10 >= 7)
                if use_shift:
                    bx0, bx1, by0, by1 = self._shift_bbox(x_min, x_max, y_min, y_max, S, S, index)
                else:
                    bx0, bx1, by0, by1 = self._zoom_out_bbox(x_min, x_max, y_min, y_max, S, S)
            else:
                bx0, bx1, by0, by1 = x_min, x_max, y_min, y_max
            boxes = torch.tensor([[bx0, by0, bx1, by1]], dtype=torch.float)
            point_coords, point_labels = init_point_sampling(mask, self.point_num)
        else:
            key          = mask_path.split('/')[-1]
            boxes        = torch.as_tensor(self.prompt_list[key]["boxes"], dtype=torch.float)
            point_coords = torch.as_tensor(self.prompt_list[key]["point_coords"], dtype=torch.float)
            point_labels = torch.as_tensor(self.prompt_list[key]["point_labels"], dtype=torch.int)

        image_input = {
            "image":         image,
            "label":         mask.unsqueeze(0),
            "point_coords":  point_coords,
            "point_labels":  point_labels,
            "boxes":         boxes,
            "original_size": (h, w),
            "label_path":    '/'.join(mask_path.split('/')[:-1]),
        }
        if self.return_ori_mask:
            image_input["ori_label"] = ori_mask
        if self.requires_name:
            image_input["name"] = self.label_paths[index].split('/')[-1]
        return image_input

    def __len__(self):
        return len(self.label_paths)


class TrainingDataset(Dataset):
    def __init__(self, data_dir, image_size=256, mode='train', requires_name=True,
                 point_num=1, mask_num=5, zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        self.image_size    = image_size
        self.requires_name = requires_name
        self.point_num     = point_num
        self.mask_num      = mask_num
        self.zoom_ratio    = zoom_ratio
        self.shift_ratio   = shift_ratio
        self.pixel_mean    = [123.675, 116.28, 103.53]
        self.pixel_std     = [58.395, 57.12, 57.375]
        dataset = json.load(open(os.path.join(data_dir, f'image2label_{mode}.json')))
        self.image_paths = list(dataset.keys())
        self.label_paths = list(dataset.values())

    def _noisy_box(self, mask_tensor):
        y_idx, x_idx = torch.where(mask_tensor > 0)
        if len(y_idx) == 0:
            return torch.zeros(1, 4, dtype=torch.float)
        x0, x1 = int(x_idx.min()), int(x_idx.max())
        y0, y1 = int(y_idx.min()), int(y_idx.max())
        S  = self.image_size
        lo, hi = self.zoom_ratio
        gw, gh = x1 - x0, y1 - y0
        bx0 = max(0, x0 - gw * random.uniform(lo, hi))
        bx1 = min(S, x1 + gw * random.uniform(lo, hi))
        by0 = max(0, y0 - gh * random.uniform(lo, hi))
        by1 = min(S, y1 + gh * random.uniform(lo, hi))
        return torch.tensor([[bx0, by0, bx1, by1]], dtype=torch.float)

    def __getitem__(self, index):
        image  = cv2.imread(self.image_paths[index])
        image  = (image - self.pixel_mean) / self.pixel_std
        h, w, _ = image.shape
        transforms = train_transforms(self.image_size, h, w)
        masks_list, boxes_list, pc_list, pl_list = [], [], [], []
        for m_path in random.choices(self.label_paths[index], k=self.mask_num):
            pre_mask = cv2.imread(m_path, 0)
            if pre_mask.max() == 255: pre_mask = pre_mask / 255
            aug = transforms(image=image, mask=pre_mask)
            img_t, mask_t = aug['image'], aug['mask'].to(torch.int64)
            boxes = self._noisy_box(mask_t)
            pc, pl = init_point_sampling(mask_t, self.point_num)
            masks_list.append(mask_t); boxes_list.append(boxes)
            pc_list.append(pc);        pl_list.append(pl)
        image_input = {
            "image":        img_t.unsqueeze(0),
            "label":        torch.stack(masks_list).unsqueeze(1),
            "boxes":        torch.stack(boxes_list),
            "point_coords": torch.stack(pc_list),
            "point_labels": torch.stack(pl_list),
        }
        if self.requires_name:
            image_input["name"] = self.image_paths[index].split('/')[-1]
        return image_input

    def __len__(self):
        return len(self.image_paths)


def stack_dict_batched(batched_input):
    out_dict = {}
    for k, v in batched_input.items():
        if isinstance(v, list): out_dict[k] = v
        else: out_dict[k] = v.reshape(-1, *v.shape[2:])
    return out_dict

In [ ]:
%%writefile /kaggle/working/SAM-Med2D/metrics.py
import torch
import torch.nn as nn


class SegMetrics(nn.Module):
    def __init__(self, pred, label, metrics):
        super().__init__()
        pred_bin  = (pred  > 0).float()
        label_bin = (label > 0).float()
        B = pred_bin.shape[0]
        p = pred_bin.view(B, -1)
        g = label_bin.view(B, -1)
        tp  = (p * g).sum(dim=1)
        fp  = (p * (1 - g)).sum(dim=1)
        fn  = ((1 - p) * g).sum(dim=1)
        eps = 1e-7
        self._map = {
            'iou':       (tp / (tp + fp + fn + eps)).mean(),
            'dice':      (2 * tp / (2 * tp + fp + fn + eps)).mean(),
            'precision': (tp / (tp + fp + eps)).mean(),
            'recall':    (tp / (tp + fn + eps)).mean(),
        }
        self._results = [self._map[m] for m in metrics]

    def __getitem__(self, idx): return self._results[idx]
    def __len__(self): return len(self._results)

In [ ]:
%%writefile /kaggle/working/SAM-Med2D/test.py
from segment_anything import sam_model_registry
import torch, torch.nn as nn, argparse, os, csv, json, random
import numpy as np
from torch.nn import functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import logging
import datetime
from scipy.ndimage import binary_erosion, distance_transform_edt
from utils import FocalDiceloss_IoULoss, generate_point, save_masks
from DataLoader import TestingDataset
from metrics import SegMetrics


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir",       type=str,   default="workdir")
    parser.add_argument("--run_name",       type=str,   default="sammed")
    parser.add_argument("--batch_size",     type=int,   default=1)
    parser.add_argument("--image_size",     type=int,   default=256)
    parser.add_argument("--device",         type=str,   default="cuda")
    parser.add_argument("--data_path",      type=str,   default="data_demo")
    parser.add_argument("--metrics",        nargs="+",  default=["iou","dice","precision","recall"])
    parser.add_argument("--model_type",     type=str,   default="vit_b")
    parser.add_argument("--sam_checkpoint", type=str,   default="pretrain_model/sam-med2d_b.pth")
    parser.add_argument("--boxes_prompt",   type=bool,  default=True)
    parser.add_argument("--point_num",      type=int,   default=1)
    parser.add_argument("--iter_point",     type=int,   default=1)
    parser.add_argument("--multimask",      type=bool,  default=True)
    parser.add_argument("--encoder_adapter",type=bool,  default=True)
    parser.add_argument("--prompt_path",    type=str,   default=None)
    parser.add_argument("--save_pred",      type=bool,  default=False)
    parser.add_argument("--prompt_mode",    type=str,   default="zoom_out",
                        choices=["zoom_out","shift","mixed_7_3"])
    parser.add_argument("--zoom_ratio",     type=float, nargs=2, default=[0.15, 0.45])
    parser.add_argument("--shift_ratio",    type=float, default=0.30)
    args = parser.parse_args()
    if args.iter_point > 1:
        args.point_num = 1
    return args


def to_device(batch_input, device):
    out = {}
    for k, v in batch_input.items():
        if v is None:
            out[k] = v
        elif k in ("image", "label"):
            out[k] = v.float().to(device)
        elif isinstance(v, (list, torch.Size)):
            out[k] = v
        else:
            out[k] = v.to(device)
    return out


def postprocess_masks(low_res_masks, image_size, original_size):
    ori_h, ori_w = original_size
    masks = F.interpolate(low_res_masks, (image_size, image_size),
                          mode="bilinear", align_corners=False)
    if ori_h < image_size and ori_w < image_size:
        top  = torch.div(image_size - ori_h, 2, rounding_mode="trunc")
        left = torch.div(image_size - ori_w, 2, rounding_mode="trunc")
        masks = masks[..., top:ori_h+top, left:ori_w+left]
        pad = (top, left)
    else:
        masks = F.interpolate(masks, original_size, mode="bilinear", align_corners=False)
        pad = None
    return masks, pad


def prompt_and_decoder(args, batched_input, model, image_embeddings):
    points = (batched_input["point_coords"], batched_input["point_labels"]) \
             if batched_input["point_coords"] is not None else None
    with torch.no_grad():
        sparse_emb, dense_emb = model.prompt_encoder(
            points=points,
            boxes=batched_input.get("boxes"),
            masks=batched_input.get("mask_inputs"),
        )
        low_res_masks, iou_preds = model.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=args.multimask,
        )
    if args.multimask:
        max_vals, max_idx = torch.max(iou_preds, dim=1)
        iou_preds     = max_vals.unsqueeze(1)
        low_res_masks = torch.stack([low_res_masks[i:i+1, idx]
                                     for i, idx in enumerate(max_idx)], 0)
    masks = F.interpolate(low_res_masks, (args.image_size, args.image_size),
                          mode="bilinear", align_corners=False)
    return masks, low_res_masks, iou_preds


def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return 256.0
    pe = pred ^ binary_erosion(pred)
    ge = gt   ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return 256.0
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))


def calc_cbl(pred_bin, gt_bin):
    if gt_bin.sum() == 0: return None
    ys, xs  = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d / gt_diag, 0.0, 1.0))


def main(args):
    print('*'*80)
    for k, v in vars(args).items(): print(f"  {k}: {v}")
    print('*'*80)

    model = sam_model_registry[args.model_type](args).to(args.device)
    criterion = FocalDiceloss_IoULoss()
    test_dataset = TestingDataset(
        data_path=args.data_path, image_size=args.image_size,
        mode='test', requires_name=True, point_num=args.point_num,
        return_ori_mask=True, prompt_path=args.prompt_path,
        prompt_mode=args.prompt_mode,
        zoom_ratio=tuple(args.zoom_ratio), shift_ratio=args.shift_ratio)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)
    print(f'Test data: {len(test_loader)}  |  prompt_mode: {args.prompt_mode}')

    model.eval()
    test_loss, test_hd95, test_cbl = [], [], []
    test_iter_metrics = [0.0] * len(args.metrics)
    n = len(test_loader)

    save_path = os.path.join(args.work_dir, args.run_name, f"boxes_{args.prompt_mode}")

    for batched_input in tqdm(test_loader):
        batched_input = to_device(batched_input, args.device)
        ori_labels    = batched_input["ori_label"]
        original_size = batched_input["original_size"]
        img_name      = batched_input['name'][0]

        with torch.no_grad():
            image_embeddings = model.image_encoder(batched_input["image"])

        batched_input["point_coords"] = None
        batched_input["point_labels"] = None
        masks, low_res_masks, iou_preds = prompt_and_decoder(
            args, batched_input, model, image_embeddings)

        masks, pad = postprocess_masks(low_res_masks, args.image_size, original_size)
        if args.save_pred:
            save_masks(masks, save_path, img_name, args.image_size,
                       original_size, pad, batched_input.get("boxes"), None)

        loss = criterion(masks, ori_labels, iou_preds)
        test_loss.append(loss.item())

        pred_bin = (masks[0,0].cpu().numpy() > 0).astype(np.float32)
        gt_bin   = ori_labels[0,0].cpu().numpy().astype(np.float32)
        test_hd95.append(calc_hd95(pred_bin, gt_bin))
        cbl = calc_cbl(pred_bin, gt_bin)
        if cbl is not None: test_cbl.append(cbl)

        bm = SegMetrics(masks, ori_labels, args.metrics)
        for j in range(len(args.metrics)):
            test_iter_metrics[j] += float(bm[j])

    avg = {args.metrics[i]: test_iter_metrics[i]/n for i in range(len(args.metrics))}
    mean_hd95 = float(np.mean(test_hd95)) if test_hd95 else 0.0
    mean_cbl  = float(np.mean(test_cbl))  if test_cbl  else 0.0
    print(f"\n[{args.prompt_mode}] loss={np.mean(test_loss):.4f} | "
          f"IoU={avg.get('iou',0):.4f} | Dice={avg.get('dice',0):.4f} | "
          f"Pre={avg.get('precision',0):.4f} | Rec={avg.get('recall',0):.4f} | "
          f"HD95={mean_hd95:.2f}px | CBL={mean_cbl:.4f} | N={n}")

    os.makedirs(os.path.join(args.work_dir, "csv"), exist_ok=True)
    csv_path = os.path.join(args.work_dir, "csv", f"sammed2d_{args.prompt_mode}.csv")
    with open(csv_path, "w", newline="") as fc:
        w = csv.writer(fc)
        w.writerow(["model","prompt_mode","dice","iou","precision","recall","hd95","cbl","n"])
        w.writerow(["SAM-Med2D-FT", args.prompt_mode,
                    f"{avg.get('dice',0):.4f}", f"{avg.get('iou',0):.4f}",
                    f"{avg.get('precision',0):.4f}", f"{avg.get('recall',0):.4f}",
                    f"{mean_hd95:.4f}", f"{mean_cbl:.4f}", n])
    print(f"CSV saved: {csv_path}")


if __name__ == '__main__':
    main(parse_args())


### 2.1 – Kịch bản A: Zoom-Out (~30%)
Bbox mở rộng đều 4 phía 30% so với tight bbox GT.

In [ ]:
assert FT_CHECKPOINT, "❌ FT_CHECKPOINT rỗng — chạy cell Chuẩn bị (Phần 2) trước"
!python test.py \
    --work_dir  workdir/test_results \
    --data_path dataset_BTXRD \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --encoder_adapter True \
    --prompt_mode zoom_out \
    --sam_checkpoint {FT_CHECKPOINT} \
    --save_pred True

### 2.2 – Kịch bản B: Shift (~30%)
Bbox lệch vị trí ~30% — mô phỏng người dùng chú thích không chính xác.

In [ ]:
assert FT_CHECKPOINT, "❌ FT_CHECKPOINT rỗng — chạy cell Chuẩn bị (Phần 2) trước"
!python test.py \
    --work_dir  workdir/test_results \
    --data_path dataset_BTXRD \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --encoder_adapter True \
    --prompt_mode shift \
    --sam_checkpoint {FT_CHECKPOINT} \
    --save_pred True

### 2.3 – Kịch bản C: Mixed 70/30
70% zoom-out + 30% shift — kịch bản thực tế người dùng.

In [ ]:
assert FT_CHECKPOINT, "❌ FT_CHECKPOINT rỗng — chạy cell Chuẩn bị (Phần 2) trước"
!python test.py \
    --work_dir  workdir/test_results \
    --data_path dataset_BTXRD \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --encoder_adapter True \
    --prompt_mode mixed_7_3 \
    --sam_checkpoint {FT_CHECKPOINT} \
    --save_pred True

### 2.4 – Tổng hợp kết quả & Visualization
Bảng so sánh 3 kịch bản + hiển thị trực quan PGA-style.

In [ ]:
import cv2, numpy as np, os, csv, json as _json, glob as _glob
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from IPython.display import display as _ipy_display
from scipy.ndimage import binary_erosion, distance_transform_edt

%cd /kaggle/working/SAM-Med2D

MODES     = ['zoom_out', 'shift', 'mixed_7_3']
PRED_DIRS = {m: f"workdir/test_results/sammed/boxes_{m}" for m in MODES}
MASK_DIR  = "dataset_BTXRD/test/masks"
S = 256

def calc_m(prob, gt, eps=1e-6):
    pm = (prob > 0.5).astype(np.float32)
    gm = (gt   > 0.5).astype(np.float32)
    tp = (pm*gm).sum(); fp = (pm*(1-gm)).sum(); fn = ((1-pm)*gm).sum()
    p, g = pm.astype(bool), gm.astype(bool)
    hd95 = float(S)
    if p.any() and g.any():
        pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
        d1 = distance_transform_edt(~ge)[pe]
        d2 = distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2):
            hd95 = float(max(np.percentile(d1, 95), np.percentile(d2, 95)))
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        d = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean()-xs.mean())**2 +
                                          (yp.mean()-ys.mean())**2) / d, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),
                iou =float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),
                recall   =float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS = ['dice','iou','precision','recall','hd95','cbl']
HDRS = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']

# ── 1. Metrics image-level ────────────────────────────────────────────────────
print(f'\n{"="*78}')
print('  SAM-Med2D Fine-tuned — IMAGE-LEVEL metrics (GT union + max-merge)')
print(f'{"="*78}')
print(f"  {'Mode':<16}" + ''.join(f'{h:>8}' for h in HDRS) + f"  {'N_img':>6}")
print(f"  {'-'*74}")

os.makedirs('workdir/test_results/csv', exist_ok=True)
all_img_recs = {}
ft_results = {}

with open('dataset_BTXRD/label2image_test.json') as _f:
    l2i = _json.load(_f)
img_lookup = {}
for mp, ip in l2i.items():
    st = os.path.basename(mp).rsplit('_', 1)[0]
    if st not in img_lookup: img_lookup[st] = ip

for mode in MODES:
    pred_dir = PRED_DIRS[mode]
    if not os.path.exists(pred_dir):
        print(f'  ⚠ {mode}: chưa có prediction — chạy cell Test trước'); continue

    stem_data = {}
    for fn in sorted(os.listdir(MASK_DIR)):
        if not fn.endswith('.png'): continue
        parts = fn.rsplit('_', 1)
        if len(parts) < 2: continue
        stem = parts[0]
        gt_p = cv2.imread(os.path.join(MASK_DIR,   fn), cv2.IMREAD_GRAYSCALE)
        pr_p = cv2.imread(os.path.join(pred_dir,   fn), cv2.IMREAD_GRAYSCALE)
        if gt_p is None or pr_p is None: continue
        gt256 = cv2.resize(gt_p, (S,S), interpolation=cv2.INTER_NEAREST).astype(np.float32)/255.
        pr256 = cv2.resize(pr_p, (S,S), interpolation=cv2.INTER_NEAREST).astype(np.float32)/255.
        if stem not in stem_data:
            stem_data[stem] = dict(gt=gt256.copy(), prob=pr256.copy(), n=1)
        else:
            np.maximum(stem_data[stem]['gt'],   gt256, out=stem_data[stem]['gt'])
            np.maximum(stem_data[stem]['prob'], pr256, out=stem_data[stem]['prob'])
            stem_data[stem]['n'] += 1

    if not stem_data: continue
    recs = []
    for stem, d in sorted(stem_data.items()):
        m = calc_m(d['prob'], d['gt'])
        recs.append(dict(stem=stem, gt=d['gt'], prob=d['prob'], n=d['n'],
                         img_path=img_lookup.get(stem,''), **m))
    all_img_recs[mode] = recs

    avg = {k: np.mean([r[k] for r in recs]) for k in KEYS}
    ft_results[mode] = avg
    print(f"  {mode:<16}" + ''.join(f"{avg[k]:>8.4f}" for k in KEYS) + f"  {len(recs):>6}")

    csv_path = f'workdir/test_results/csv/sammed2d_finetune_{mode}_img.csv'
    with open(csv_path, 'w', newline='') as fc:
        w = csv.writer(fc)
        w.writerow(['model','mode'] + KEYS + ['N_img'])
        w.writerow(['SAM-Med2D-FT', mode] + [f'{avg[k]:.4f}' for k in KEYS] + [len(recs)])

print(f'{"="*78}')

# ── 2. PGA-style 5-col visualization ──────────────────────────────────────────
VIS_MODE = 'zoom_out'
MIN_POLY  = 2
N_MIN     = 10

if VIS_MODE not in all_img_recs:
    print(f"⚠ Chưa có kết quả {VIS_MODE} — chạy Test A trước")
else:
    recs_viz = [r for r in all_img_recs[VIS_MODE] if r['n'] >= MIN_POLY]
    assert len(recs_viz) >= N_MIN, f'Chỉ có {len(recs_viz)} ảnh ≥{MIN_POLY} GT (cần {N_MIN})'
    N_SHOW = len(recs_viz)

    def get_poly_bboxes(stem, r=0.30):
        bboxes = []
        for pm_path in sorted(_glob.glob(os.path.join(MASK_DIR, f"{stem}_*.png"))):
            pm = cv2.imread(pm_path, cv2.IMREAD_GRAYSCALE)
            if pm is None: continue
            pm256 = cv2.resize(pm, (S,S), interpolation=cv2.INTER_NEAREST).astype(np.float32)
            if pm256.max() > 1: pm256 /= 255.
            ys, xs = np.where(pm256 > 0.5)
            if not len(xs): continue
            x0,x1 = int(xs.min()),int(xs.max())
            y0,y1 = int(ys.min()),int(ys.max())
            gw,gh = x1-x0, y1-y0
            bboxes.append((max(0,x0-gw*r), max(0,y0-gh*r),
                           min(S,x1+gw*r)-max(0,x0-gw*r),
                           min(S,y1+gh*r)-max(0,y0-gh*r)))
        return bboxes

    fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4*N_SHOW))
    if N_SHOW == 1: axes = axes[np.newaxis, :]
    fig.suptitle(
        f'SAM-Med2D Fine-tuned — {N_SHOW} ảnh có ≥{MIN_POLY} GT polygon (zoom_out, image-level)',
        fontsize=13, y=1.001)

    for ax, ct in zip(axes[0], ['Ảnh gốc','Ảnh + Bbox Prompts',
                                  'Dự đoán (merged)','Ground Truth','TP/FP/FN']):
        ax.set_title(ct, fontsize=10, fontweight='bold')

    for count, rec in enumerate(recs_viz):
        img_gray = cv2.imread(rec['img_path'], cv2.IMREAD_GRAYSCALE) if rec['img_path'] else None
        if img_gray is not None:
            img_np = cv2.resize(img_gray, (S,S)).astype(np.float32) / 255.
        else:
            img_np = np.zeros((S,S), dtype=np.float32)

        gt_np   = (rec['gt']   > 0.5).astype(float)
        pred_np = (rec['prob'] > 0.5).astype(float)
        e   = 1e-6
        tp  = (pred_np*gt_np).sum()
        fp  = (pred_np*(1-gt_np)).sum()
        fn  = ((1-pred_np)*gt_np).sum()
        dice = float((2*tp+e)/(2*tp+fp+fn+e))
        iou  = float((tp+e)/(tp+fp+fn+e))
        pre  = float((tp+e)/(tp+fp+e))
        rec_ = float((tp+e)/(tp+fn+e))

        row = axes[count]
        bg  = np.stack([img_np]*3, axis=-1)

        # Col 0 – Ảnh gốc
        row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        row[0].set_ylabel(f'#{count+1} [{rec["n"]} poly]\nDice={dice:.3f}', fontsize=8)

        # Col 1 – Ảnh + per-polygon bbox prompts
        row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
        for bx in get_poly_bboxes(rec['stem']):
            row[1].add_patch(Rectangle((bx[0],bx[1]), bx[2], bx[3],
                             linewidth=2, edgecolor='yellow', facecolor='none'))
        row[1].set_title(rec['stem'], fontsize=6, pad=2)

        # Col 2 – Dự đoán (red overlay)
        pr_ov = bg.copy()
        pr_ov[...,0] = np.clip(pr_ov[...,0] + pred_np*0.55, 0, 1)
        pr_ov[...,1] = np.clip(pr_ov[...,1] - pred_np*0.20, 0, 1)
        row[2].imshow(pr_ov)
        row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

        # Col 3 – Ground Truth (green overlay)
        gt_ov = bg.copy()
        gt_ov[...,1] = np.clip(gt_ov[...,1] + gt_np*0.55, 0, 1)
        gt_ov[...,0] = np.clip(gt_ov[...,0] - gt_np*0.20, 0, 1)
        row[3].imshow(gt_ov)

        # Col 4 – TP(green) / FP(red) / FN(blue)
        inter = bg.copy() * 0.25
        inter[...,1] = np.clip(inter[...,1] + pred_np*gt_np      *0.9, 0, 1)
        inter[...,0] = np.clip(inter[...,0] + pred_np*(1-gt_np)  *1.0, 0, 1)
        inter[...,2] = np.clip(inter[...,2] + (1-pred_np)*gt_np  *1.0, 0, 1)
        row[4].imshow(inter)
        row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7,
                         color='saddlebrown', pad=2)

        for ax in row: ax.axis('off')

    fig.legend(
        handles=[Patch(facecolor='green', label='TP'),
                 Patch(facecolor='red',   label='FP'),
                 Patch(facecolor='blue',  label='FN')],
        loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
    )
    plt.tight_layout()
    _ipy_display(fig)
    plt.close(fig)

---
# PHẦN 3 — SAM-Med2D Zero-Shot
---

## SAM-Med2D — Zero-Shot Test (không fine-tune)

So sánh SAM-Med2D **pre-trained** (gốc, không fine-tune trên BTXRD) vs **fine-tuned** vs **PGA-UNet**.

Chạy tuần tự: Setup → Checkpoint → WriteFiles → Label JSON → Test × 3 → Kết quả

In [ ]:
%cd /kaggle/working
import os

# ── Repo: Phần 2 đã clone → chỉ clone nếu chưa có ───────────────────────────
if not os.path.exists('SAM-Med2D'):
    !git clone -q https://github.com/OpenGVLab/SAM-Med2D/
else:
    print('[Skip] SAM-Med2D repo đã tồn tại từ Phần 2 — bỏ qua clone')

%cd /kaggle/working/SAM-Med2D
!pip install -q albumentations gdown

# ── Dataset: Phần 2 đã download → chỉ download nếu chưa có ──────────────────
if not os.path.exists('dataset_BTXRD/test/images'):
    import gdown as _gdown
    _gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                    '/tmp/dataset_BTXRD_zs.zip', quiet=False)
    !unzip -oq /tmp/dataset_BTXRD_zs.zip -d /kaggle/working/SAM-Med2D/
else:
    print('[Skip] dataset_BTXRD đã tồn tại từ Phần 2 — bỏ qua download')

print('\n✅ ZS Setup done!')


In [ ]:
# ── Chuyển đổi: JSON annotation → per-polygon mask PNG ──────────────────────
# Mỗi polygon (khối u) → 1 file mask riêng (IMG001768_1.png, IMG001768_2.png...)
# Sau bước này: train=1859 samples, val=211, test=248 (khớp chính xác với PGA)

import cv2, json, numpy as np, os, glob

%cd /kaggle/working/SAM-Med2D

def build_per_polygon_masks(split):
    ann_dir  = f"dataset_BTXRD/{split}/annotations"
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"

    if not os.path.exists(ann_dir):
        print(f"[!] {split}: không có thư mục annotations, bỏ qua."); return

    # Xóa toàn bộ mask cũ (merged 1-per-image) để tránh đếm nhầm
    old = glob.glob(f"{mask_dir}/*.png")
    for p in old: os.remove(p)
    print(f"[*] {split}: đã xóa {len(old)} mask cũ")

    n_written = 0
    for json_path in sorted(glob.glob(f"{ann_dir}/*.json")):
        base = os.path.splitext(os.path.basename(json_path))[0]

        # Tìm file ảnh gốc (.png / .jpg)
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg']:
            p = os.path.join(img_dir, base + ext)
            if os.path.exists(p): img_path = p; break
        if img_path is None:
            print(f"  [!] Không tìm thấy ảnh cho {base}"); continue

        img  = cv2.imread(img_path)
        H, W = img.shape[:2]
        data = json.load(open(json_path))

        polygons = [s for s in data.get('shapes', [])
                    if s.get('shape_type') == 'polygon']

        for i, shape in enumerate(polygons, start=1):
            pts  = np.array(shape['points'], dtype=np.float32)
            pts  = pts.reshape((-1, 1, 2)).astype(np.int32)
            mask = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            out  = os.path.join(mask_dir, f"{base}_{i}.png")
            cv2.imwrite(out, mask)
            n_written += 1

    print(f"[✅] {split}: tạo {n_written} per-polygon mask files")

for split in ['train', 'val', 'test']:
    build_per_polygon_masks(split)

print("\n✅ Xong! Số samples mới:")
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f"dataset_BTXRD/{split}/masks/*.png"))
    print(f"  {split}: {n} mask files")


In [ ]:
# Tải SAM-Med2D PRE-TRAINED checkpoint (KHÔNG fine-tune)
import os
os.makedirs('checkpoint', exist_ok=True)
!gdown 'https://drive.google.com/uc?id=1ARiB5RkSsWmAB_8mqWnwDF8ZKTtFwsjl' \
    -O checkpoint/sam_med2d.pth
!ls -lh checkpoint/
ZS_CHECKPOINT = 'checkpoint/sam_med2d.pth'
assert os.path.exists(ZS_CHECKPOINT), '❌ Tải thất bại'
print(f'✅ Zero-shot checkpoint: {ZS_CHECKPOINT}  ({os.path.getsize(ZS_CHECKPOINT)//1024//1024} MB)')


In [ ]:
%%writefile /kaggle/working/SAM-Med2D/segment_anything/build_sam.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.

# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

import torch
from functools import partial
from .modeling import ImageEncoderViT, MaskDecoder, PromptEncoder, Sam, TwoWayTransformer
from torch.nn import functional as F

def build_sam_vit_h(args):
    return _build_sam(
        encoder_embed_dim=1280,
        encoder_depth=32,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[7, 15, 23, 31],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,
    )


build_sam = build_sam_vit_h


def build_sam_vit_l(args):
    return _build_sam(
        encoder_embed_dim=1024,
        encoder_depth=24,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[5, 11, 17, 23],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,
    )


def build_sam_vit_b(args):
    return _build_sam(
        encoder_embed_dim=768,
        encoder_depth=12,
        encoder_num_heads=12,
        encoder_global_attn_indexes=[2, 5, 8, 11],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,

    )


sam_model_registry = {
    "default": build_sam_vit_h,
    "vit_h": build_sam_vit_h,
    "vit_l": build_sam_vit_l,
    "vit_b": build_sam_vit_b,
}


def _build_sam(
    encoder_embed_dim,
    encoder_depth,
    encoder_num_heads,
    encoder_global_attn_indexes,
    image_size,
    checkpoint,
    encoder_adapter,
):
    prompt_embed_dim = 256
    image_size = image_size
    vit_patch_size = 16
    image_embedding_size = image_size // vit_patch_size
    sam = Sam(
        image_encoder=ImageEncoderViT(
            depth=encoder_depth,
            embed_dim=encoder_embed_dim,
            img_size=image_size,
            mlp_ratio=4,
            norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
            num_heads=encoder_num_heads,
            patch_size=vit_patch_size,
            qkv_bias=True,
            use_rel_pos = True,
            global_attn_indexes=encoder_global_attn_indexes,
            window_size=14,
            out_chans=prompt_embed_dim,
            adapter_train = encoder_adapter,
        ),
        prompt_encoder=PromptEncoder(
            embed_dim=prompt_embed_dim,
            image_embedding_size=(image_embedding_size, image_embedding_size),
            input_image_size=(image_size, image_size),
            mask_in_chans=16,
        ),
        mask_decoder=MaskDecoder(
            num_multimask_outputs=3,
            transformer=TwoWayTransformer(
                depth=2,
                embedding_dim=prompt_embed_dim,
                mlp_dim=2048,
                num_heads=8,
            ),
            transformer_dim=prompt_embed_dim,
            iou_head_depth=3,
            iou_head_hidden_dim=256,
        ),
        pixel_mean=[123.675, 116.28, 103.53],
        pixel_std=[58.395, 57.12, 57.375],
    )
    # sam.train()
    if checkpoint is not None:
        with open(checkpoint, "rb") as f:
            # Ép gán weights_only=False để PyTorch mới chịu đọc file chứa Adam optimizer của tác giả
            state_dict = torch.load(f, map_location="cpu", weights_only=False)
        try:
            if 'model' in state_dict.keys():
                print(encoder_adapter)
                sam.load_state_dict(state_dict['model'], False) # Tác giả đã để strict=False ở đây sẵn rồi
            else:
                if image_size==1024 and encoder_adapter==True:
                    sam.load_state_dict(state_dict, False)
                else:
                    sam.load_state_dict(state_dict)
        except:
            print('*******interpolate')
            new_state_dict = load_from(sam, state_dict, image_size, vit_patch_size)
            sam.load_state_dict(new_state_dict)
        print(f"*******load {checkpoint}")

    return sam


def load_from(sam, state_dicts, image_size, vit_patch_size):

    sam_dict = sam.state_dict()
    except_keys = ['mask_tokens', 'output_hypernetworks_mlps', 'iou_prediction_head']
    new_state_dict = {k: v for k, v in state_dicts.items() if
                      k in sam_dict.keys() and except_keys[0] not in k and except_keys[1] not in k and except_keys[2] not in k}
    pos_embed = new_state_dict['image_encoder.pos_embed']
    token_size = int(image_size // vit_patch_size)
    if pos_embed.shape[1] != token_size:
        # resize pos embedding, which may sacrifice the performance, but I have no better idea
        pos_embed = pos_embed.permute(0, 3, 1, 2)  # [b, c, h, w]
        pos_embed = F.interpolate(pos_embed, (token_size, token_size), mode='bilinear', align_corners=False)
        pos_embed = pos_embed.permute(0, 2, 3, 1)  # [b, h, w, c]
        new_state_dict['image_encoder.pos_embed'] = pos_embed
        rel_pos_keys = [k for k in sam_dict.keys() if 'rel_pos' in k]

        global_rel_pos_keys = [k for k in rel_pos_keys if
                                                        '2' in k or
                                                        '5' in k or
                                                        '7' in k or
                                                        '8' in k or
                                                        '11' in k or
                                                        '13' in k or
                                                        '15' in k or
                                                        '23' in k or
                                                        '31' in k]
        # print(sam_dict)
        for k in global_rel_pos_keys:
            h_check, w_check = sam_dict[k].shape
            rel_pos_params = new_state_dict[k]
            h, w = rel_pos_params.shape
            rel_pos_params = rel_pos_params.unsqueeze(0).unsqueeze(0)
            if h != h_check or w != w_check:
                rel_pos_params = F.interpolate(rel_pos_params, (h_check, w_check), mode='bilinear', align_corners=False)

            new_state_dict[k] = rel_pos_params[0, 0, ...]

    sam_dict.update(new_state_dict)
    return sam_dict

In [ ]:
%%writefile /kaggle/working/SAM-Med2D/DataLoader.py
# (Viết trước training để TrainingDataset có _noisy_box + zoom_ratio)
import os
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
import numpy as np
from torch.nn import functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import train_transforms, get_boxes_from_mask, init_point_sampling
import json
import random

class TestingDataset(Dataset):
    def __init__(self, data_path, image_size=256, mode='test', requires_name=True, point_num=1, return_ori_mask=True, prompt_path=None, prompt_mode='zoom_out', zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        """
        Initializes a TestingDataset object.
        """
        self.image_size = image_size
        self.return_ori_mask = return_ori_mask
        self.prompt_path = prompt_path
        self.prompt_list = {} if prompt_path is None else json.load(open(prompt_path, "r"))
        self.requires_name = requires_name
        self.point_num = point_num
        self.mode = mode
        self.is_train = (mode == 'train')

        # Thêm biến điều khiển robust bbox
        self.prompt_mode = prompt_mode
        self.zoom_ratio = zoom_ratio
        self.shift_ratio = shift_ratio

        json_file = open(os.path.join(data_path, f'label2image_{mode}.json'), "r")
        dataset = json.load(json_file)

        self.image_paths = list(dataset.values())
        self.label_paths = list(dataset.keys())

        self.pixel_mean = [123.675, 116.28, 103.53]
        self.pixel_std = [58.395, 57.12, 57.375]

    def _zoom_out_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        lo, hi = self.zoom_ratio
        if self.is_train:
            r_l, r_r = random.uniform(lo, hi), random.uniform(lo, hi)
            r_t, r_b = random.uniform(lo, hi), random.uniform(lo, hi)
        else:
            r = (lo + hi) / 2
            r_l = r_r = r_t = r_b = r
        bx_min = max(0,       x_min - gt_w * r_l)
        bx_max = min(orig_w,  x_max + gt_w * r_r)
        by_min = max(0,       y_min - gt_h * r_t)
        by_max = min(orig_h,  y_max + gt_h * r_b)
        return bx_min, bx_max, by_min, by_max

    def _shift_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=None):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)

        if self.is_train:
            dx = random.uniform(-gt_w * self.shift_ratio, gt_w * self.shift_ratio)
            dy = random.uniform(-gt_h * self.shift_ratio, gt_h * self.shift_ratio)
        else:
            rng = random.Random(seed_idx or 0)
            dx = rng.uniform(gt_w * 0.4, gt_w * 0.7) * self.shift_ratio
            dy = rng.uniform(gt_h * 0.1, gt_h * 0.3) * self.shift_ratio

        bx_min = max(0,       bx_min + dx)
        bx_max = min(orig_w,  bx_max + dx)
        by_min = max(0,       by_min + dy)
        by_max = min(orig_h,  by_max + dy)

        if min(bx_max, x_max) - max(bx_min, x_min) < gt_w * 0.3:
            if dx > 0: bx_max = min(orig_w, x_max + gt_w * 0.15)
            else:      bx_min = max(0,      x_min - gt_w * 0.15)
        if min(by_max, y_max) - max(by_min, y_min) < gt_h * 0.3:
            if dy > 0: by_max = min(orig_h, y_max + gt_h * 0.15)
            else:      by_min = max(0,      y_min - gt_h * 0.15)
        return bx_min, bx_max, by_min, by_max

    def __getitem__(self, index):
        image_input = {}
        try:
            image = cv2.imread(self.image_paths[index])
            image = (image - self.pixel_mean) / self.pixel_std
        except:
            print(self.image_paths[index])

        mask_path = self.label_paths[index]
        ori_np_mask = cv2.imread(mask_path, 0)

        if ori_np_mask.max() == 255:
            ori_np_mask = ori_np_mask / 255

        assert np.array_equal(ori_np_mask, ori_np_mask.astype(bool)), f"Mask should only contain binary values 0 and 1. {self.label_paths[index]}"

        h, w = ori_np_mask.shape
        ori_mask = torch.tensor(ori_np_mask).unsqueeze(0)

        transforms = train_transforms(self.image_size, h, w)
        augments = transforms(image=image, mask=ori_np_mask)
        image, mask = augments['image'], augments['mask'].to(torch.int64)

        if self.prompt_path is None:
            # Lấy bbox chặt từ mask đã transform
            y_indices, x_indices = torch.where(mask > 0)
            if len(y_indices) > 0 and len(x_indices) > 0:
                y_min, y_max = y_indices.min().item(), y_indices.max().item()
                x_min, x_max = x_indices.min().item(), x_indices.max().item()
            else:
                x_min, x_max, y_min, y_max = 0, self.image_size, 0, self.image_size

            orig_h, orig_w = self.image_size, self.image_size

            # Áp dụng các kịch bản nhiễu Bounding Box
            if self.prompt_mode == 'zoom_out':
                bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
            elif self.prompt_mode == 'shift':
                bx_min, bx_max, by_min, by_max = self._shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=index)
            elif self.prompt_mode == 'mixed_7_3':
                use_shift = (random.random() < 0.3) if self.is_train else (index % 10 >= 7)
                if use_shift:
                    bx_min, bx_max, by_min, by_max = self._shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=index)
                else:
                    bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
            else:
                bx_min, bx_max, by_min, by_max = x_min, x_max, y_min, y_max

            boxes = torch.tensor([[bx_min, by_min, bx_max, by_max]], dtype=torch.float)
            point_coords, point_labels = init_point_sampling(mask, self.point_num)
        else:
            prompt_key = mask_path.split('/')[-1]
            boxes = torch.as_tensor(self.prompt_list[prompt_key]["boxes"], dtype=torch.float)
            point_coords = torch.as_tensor(self.prompt_list[prompt_key]["point_coords"], dtype=torch.float)
            point_labels = torch.as_tensor(self.prompt_list[prompt_key]["point_labels"], dtype=torch.int)

        image_input["image"] = image
        image_input["label"] = mask.unsqueeze(0)
        image_input["point_coords"] = point_coords
        image_input["point_labels"] = point_labels
        image_input["boxes"] = boxes
        image_input["original_size"] = (h, w)
        image_input["label_path"] = '/'.join(mask_path.split('/')[:-1])

        if self.return_ori_mask:
            image_input["ori_label"] = ori_mask

        image_name = self.label_paths[index].split('/')[-1]
        if self.requires_name:
            image_input["name"] = image_name
            return image_input
        else:
            return image_input

    def __len__(self):
        return len(self.label_paths)

class TrainingDataset(Dataset):
    def __init__(self, data_dir, image_size=256, mode='train', requires_name=True, point_num=1, mask_num=5,
                 zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        self.image_size = image_size
        self.requires_name = requires_name
        self.point_num = point_num
        self.mask_num = mask_num
        self.zoom_ratio = zoom_ratio      # noisy bbox – cùng setup với PGA
        self.shift_ratio = shift_ratio
        self.pixel_mean = [123.675, 116.28, 103.53]
        self.pixel_std = [58.395, 57.12, 57.375]

        dataset = json.load(open(os.path.join(data_dir, f'image2label_{mode}.json'), "r"))
        self.image_paths = list(dataset.keys())
        self.label_paths = list(dataset.values())

    def _noisy_box(self, mask_tensor: torch.Tensor) -> torch.Tensor:
        """Tight bbox + zoom-out ngẫu nhiên [lo, hi] – giống PGA training."""
        y_idx, x_idx = torch.where(mask_tensor > 0)
        if len(y_idx) == 0:
            return torch.zeros(1, 4, dtype=torch.float)
        x0, x1 = int(x_idx.min()), int(x_idx.max())
        y0, y1 = int(y_idx.min()), int(y_idx.max())
        H = W = self.image_size
        lo, hi = self.zoom_ratio
        r_l = random.uniform(lo, hi); r_r = random.uniform(lo, hi)
        r_t = random.uniform(lo, hi); r_b = random.uniform(lo, hi)
        gw, gh = x1 - x0, y1 - y0
        bx0 = max(0, x0 - gw*r_l);  bx1 = min(W, x1 + gw*r_r)
        by0 = max(0, y0 - gh*r_t);  by1 = min(H, y1 + gh*r_b)
        return torch.tensor([[bx0, by0, bx1, by1]], dtype=torch.float)

    def __getitem__(self, index):
        image_input = {}
        try:
            image = cv2.imread(self.image_paths[index])
            image = (image - self.pixel_mean) / self.pixel_std
        except:
            print(self.image_paths[index])

        h, w, _ = image.shape
        transforms = train_transforms(self.image_size, h, w)

        masks_list = []
        boxes_list = []
        point_coords_list, point_labels_list = [], []
        mask_path = random.choices(self.label_paths[index], k=self.mask_num)
        for m in mask_path:
            pre_mask = cv2.imread(m, 0)
            if pre_mask.max() == 255:
                pre_mask = pre_mask / 255

            augments = transforms(image=image, mask=pre_mask)
            image_tensor, mask_tensor = augments['image'], augments['mask'].to(torch.int64)

            boxes = self._noisy_box(mask_tensor)
            point_coords, point_label = init_point_sampling(mask_tensor, self.point_num)

            masks_list.append(mask_tensor)
            boxes_list.append(boxes)
            point_coords_list.append(point_coords)
            point_labels_list.append(point_label)

        mask = torch.stack(masks_list, dim=0)
        boxes = torch.stack(boxes_list, dim=0)
        point_coords = torch.stack(point_coords_list, dim=0)
        point_labels = torch.stack(point_labels_list, dim=0)

        image_input["image"] = image_tensor.unsqueeze(0)
        image_input["label"] = mask.unsqueeze(1)
        image_input["boxes"] = boxes
        image_input["point_coords"] = point_coords
        image_input["point_labels"] = point_labels

        image_name = self.image_paths[index].split('/')[-1]
        if self.requires_name:
            image_input["name"] = image_name
            return image_input
        else:
            return image_input

    def __len__(self):
        return len(self.image_paths)

def stack_dict_batched(batched_input):
    out_dict = {}
    for k,v in batched_input.items():
        if isinstance(v, list):
            out_dict[k] = v
        else:
            out_dict[k] = v.reshape(-1, *v.shape[2:])
    return out_dict

In [ ]:
%%writefile /kaggle/working/SAM-Med2D/metrics.py
import torch
import torch.nn as nn


class SegMetrics(nn.Module):
    """
    SegMetrics(pred, label, metrics)
    pred, label : Tensor [B, 1, H, W]  (logits or 0/1)
    metrics     : list of str in {'iou','dice','precision','recall'}
    Usage       : bm = SegMetrics(pred, label, ['iou','dice','precision','recall'])
                  val = float(bm[i])
    """

    def __init__(self, pred, label, metrics):
        super().__init__()
        pred_bin  = (pred  > 0).float()
        label_bin = (label > 0).float()

        B = pred_bin.shape[0]
        p = pred_bin.view(B, -1)
        g = label_bin.view(B, -1)

        tp = (p * g).sum(dim=1)                 # [B]
        fp = (p * (1 - g)).sum(dim=1)
        fn = ((1 - p) * g).sum(dim=1)
        eps = 1e-7

        iou       = (tp / (tp + fp + fn + eps)).mean()
        dice      = (2 * tp / (2 * tp + fp + fn + eps)).mean()
        precision = (tp / (tp + fp + eps)).mean()
        recall    = (tp / (tp + fn + eps)).mean()

        self._map = {
            'iou':       iou,
            'dice':      dice,
            'precision': precision,
            'recall':    recall,
        }
        self._results = [self._map[m] for m in metrics]

    def __getitem__(self, idx):
        return self._results[idx]

    def __len__(self):
        return len(self._results)


In [ ]:
%%writefile /kaggle/working/SAM-Med2D/test.py
from segment_anything import sam_model_registry
import torch.nn as nn
import torch
import argparse
import os
from utils import FocalDiceloss_IoULoss, generate_point, save_masks
from torch.utils.data import DataLoader
from DataLoader import TestingDataset
from metrics import SegMetrics
import time
from tqdm import tqdm
import numpy as np
from torch.nn import functional as F
import logging
import datetime
import random
import csv
import json
from scipy.ndimage import binary_erosion, distance_transform_edt

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir", type=str, default="workdir", help="work dir")
    parser.add_argument("--run_name", type=str, default="sammed", help="run model name")
    parser.add_argument("--batch_size", type=int, default=1, help="batch size")
    parser.add_argument("--image_size", type=int, default=256, help="image_size")
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument("--data_path", type=str, default="data_demo", help="train data path")
    parser.add_argument("--metrics", nargs='+', default=['iou', 'dice', 'precision', 'recall'], help="metrics")
    parser.add_argument("--model_type", type=str, default="vit_b", help="sam model_type")
    parser.add_argument("--sam_checkpoint", type=str, default="pretrain_model/sam-med2d_b.pth", help="sam checkpoint")
    parser.add_argument("--boxes_prompt", type=bool, default=True, help="use boxes prompt")
    parser.add_argument("--point_num", type=int, default=1, help="point num")
    parser.add_argument("--iter_point", type=int, default=1, help="iter num")
    parser.add_argument("--multimask", type=bool, default=True, help="ouput multimask")
    parser.add_argument("--encoder_adapter", type=bool, default=True, help="use adapter")
    parser.add_argument("--prompt_path", type=str, default=None, help="fix prompt path")
    parser.add_argument("--save_pred", type=bool, default=False, help="save result")
    parser.add_argument("--prompt_mode", type=str, default="zoom_out",
                        choices=['zoom_out', 'shift', 'mixed_7_3'],
                        help="Bbox prompt scenario")
    parser.add_argument("--zoom_ratio", type=float, nargs=2, default=[0.15, 0.45])
    parser.add_argument("--shift_ratio", type=float, default=0.30)
    args = parser.parse_args()
    if args.iter_point > 1:
        args.point_num = 1
    return args

def to_device(batch_input, device):
    device_input = {}
    for key, value in batch_input.items():
        if value is not None:
            if key=='image' or key=='label':
                device_input[key] = value.float().to(device)
            elif type(value) is list or type(value) is torch.Size:
                device_input[key] = value
            else:
                device_input[key] = value.to(device)
        else:
            device_input[key] = value
    return device_input

def postprocess_masks(low_res_masks, image_size, original_size):
    ori_h, ori_w = original_size
    masks = F.interpolate(low_res_masks, (image_size, image_size),
                          mode="bilinear", align_corners=False)
    if ori_h < image_size and ori_w < image_size:
        top  = torch.div((image_size - ori_h), 2, rounding_mode='trunc')
        left = torch.div((image_size - ori_w), 2, rounding_mode='trunc')
        masks = masks[..., top:ori_h+top, left:ori_w+left]
        pad = (top, left)
    else:
        masks = F.interpolate(masks, original_size, mode="bilinear", align_corners=False)
        pad = None
    return masks, pad

def prompt_and_decoder(args, batched_input, ddp_model, image_embeddings):
    points = (batched_input["point_coords"], batched_input["point_labels"]) \
             if batched_input["point_coords"] is not None else None
    with torch.no_grad():
        sparse_embeddings, dense_embeddings = ddp_model.prompt_encoder(
            points=points,
            boxes=batched_input.get("boxes", None),
            masks=batched_input.get("mask_inputs", None),
        )
        low_res_masks, iou_predictions = ddp_model.mask_decoder(
            image_embeddings=image_embeddings,
            image_pe=ddp_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=args.multimask,
        )
    if args.multimask:
        max_values, max_indexs = torch.max(iou_predictions, dim=1)
        iou_predictions = max_values.unsqueeze(1)
        low_res_masks = torch.stack([low_res_masks[i:i+1, idx]
                                     for i, idx in enumerate(max_indexs)], 0)
    masks = F.interpolate(low_res_masks, (args.image_size, args.image_size),
                          mode="bilinear", align_corners=False)
    return masks, low_res_masks, iou_predictions

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return 256.0
    pe, ge = pred ^ binary_erosion(pred), gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return 256.0
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred_bin, gt_bin):
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d / gt_diag, 0.0, 1.0))

def main(args):
    if args.device == 'cuda' and not torch.cuda.is_available():
        print('⚠️  CUDA không khả dụng → chạy trên CPU (chậm hơn, chỉ dùng khi không có GPU)')
        args.device = 'cpu'
    print('*'*80)
    for k, v in vars(args).items(): print(f"  {k}: {v}")
    print('*'*80)

    model = sam_model_registry[args.model_type](args).to(args.device)
    criterion = FocalDiceloss_IoULoss()
    test_dataset = TestingDataset(
        data_path=args.data_path, image_size=args.image_size,
        mode='test', requires_name=True, point_num=args.point_num,
        return_ori_mask=True, prompt_path=args.prompt_path,
        prompt_mode=args.prompt_mode,
        zoom_ratio=tuple(args.zoom_ratio), shift_ratio=args.shift_ratio)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)
    print(f'Test data: {len(test_loader)}  |  prompt_mode: {args.prompt_mode}')

    model.eval()
    test_loss, test_hd95, test_cbl = [], [], []
    test_iter_metrics = [0] * len(args.metrics)
    l = len(test_loader)

    for batched_input in tqdm(test_loader):
        batched_input = to_device(batched_input, args.device)
        ori_labels   = batched_input["ori_label"]
        original_size = batched_input["original_size"]
        img_name     = batched_input['name'][0]

        with torch.no_grad():
            image_embeddings = model.image_encoder(batched_input["image"])

        batched_input["point_coords"], batched_input["point_labels"] = None, None
        save_path = os.path.join(args.work_dir, args.run_name, f"boxes_{args.prompt_mode}")
        masks, low_res_masks, iou_predictions = prompt_and_decoder(
            args, batched_input, model, image_embeddings)

        masks, pad = postprocess_masks(low_res_masks, args.image_size, original_size)
        if args.save_pred:
            save_masks(masks, save_path, img_name, args.image_size,
                       original_size, pad, batched_input.get("boxes", None), None)

        loss = criterion(masks, ori_labels, iou_predictions)
        test_loss.append(loss.item())

        pred_bin = (masks[0, 0].cpu().numpy() > 0.0).astype(np.float32)
        gt_bin   = ori_labels[0, 0].cpu().numpy().astype(np.float32)
        test_hd95.append(calc_hd95(pred_bin, gt_bin))
        cbl = calc_cbl(pred_bin, gt_bin)
        if cbl is not None: test_cbl.append(cbl)

        bm = SegMetrics(masks, ori_labels, args.metrics)
        for j in range(len(args.metrics)):
            test_iter_metrics[j] += float(bm[j])

    metrics = {args.metrics[i]: f"{test_iter_metrics[i]/l:.4f}" for i in range(len(args.metrics))}
    mean_dice = test_iter_metrics[args.metrics.index('dice')] / l if 'dice' in args.metrics else 0.0
    mean_iou  = test_iter_metrics[args.metrics.index('iou')]  / l if 'iou'  in args.metrics else 0.0
    mean_pre  = test_iter_metrics[args.metrics.index('precision')] / l if 'precision' in args.metrics else 0.0
    mean_rec  = test_iter_metrics[args.metrics.index('recall')]    / l if 'recall'    in args.metrics else 0.0
    mean_hd95 = float(np.mean(test_hd95)) if test_hd95 else 0.0
    mean_cbl  = float(np.mean(test_cbl))  if test_cbl  else 0.0
    print(f"\n[{args.prompt_mode}] loss={np.mean(test_loss):.4f} | "
          f"IoU={mean_iou:.4f} | Dice={mean_dice:.4f} | "
          f"Pre={mean_pre:.4f} | Rec={mean_rec:.4f} | "
          f"HD95={mean_hd95:.2f}px | CBL={mean_cbl:.4f} | N={l}")

    os.makedirs(os.path.join(args.work_dir, "csv"), exist_ok=True)
    csv_path = os.path.join(args.work_dir, "csv", f"sammed2d_{args.prompt_mode}.csv")
    with open(csv_path, "w", newline="") as f_csv:
        writer = csv.writer(f_csv)
        writer.writerow(["model", "prompt_mode", "dice", "iou", "precision", "recall", "hd95", "cbl", "n_samples"])
        writer.writerow(["SAM-Med2D", args.prompt_mode,
                         f"{mean_dice:.4f}", f"{mean_iou:.4f}",
                         f"{mean_pre:.4f}", f"{mean_rec:.4f}",
                         f"{mean_hd95:.4f}", f"{mean_cbl:.4f}", l])
    print(f"CSV saved: {csv_path}")

if __name__ == '__main__':
    args = parse_args()
    main(args)


In [ ]:
%cd /kaggle/working/SAM-Med2D
import os, json

def create_evaluation_json(split='test'):
    mask_dir = f'dataset_BTXRD/{split}/masks'
    img_dir  = f'dataset_BTXRD/{split}/images'
    label2image = {}
    if not os.path.exists(mask_dir) or not os.path.exists(img_dir):
        print(f'[!] {split}: không tìm thấy thư mục'); return
    img_dict = {}
    for f in os.listdir(img_dir):
        if f.lower().endswith(('.png','.jpg','.jpeg')):
            img_dict[os.path.splitext(f)[0]] = f
    for mask_name in os.listdir(mask_dir):
        if not mask_name.lower().endswith(('.png','.jpg','.jpeg')): continue
        mask_base = os.path.splitext(mask_name)[0]
        img_base  = mask_base.split('_')[0] if '_' in mask_base else mask_base
        if img_base in img_dict:
            label2image[f'dataset_BTXRD/{split}/masks/{mask_name}'] = \
                        f'dataset_BTXRD/{split}/images/{img_dict[img_base]}'
    out = f'dataset_BTXRD/label2image_{split}.json'
    with open(out,'w',encoding='utf-8') as fout:
        json.dump(label2image, fout, indent=4)
    print(f'[*] label2image_{split}.json — {len(label2image)} mẫu')

create_evaluation_json('test')
create_evaluation_json('val')
print('✅ JSON ready')


### A – Zoom-Out (~30%)

In [ ]:
assert os.path.exists(ZS_CHECKPOINT), '❌ Chạy cell Download checkpoint trước'
!python test.py \
    --work_dir 'workdir/zeroshot_results' \
    --data_path 'dataset_BTXRD' \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode zoom_out \
    --sam_checkpoint {ZS_CHECKPOINT} \
    --save_pred True


### B – Shift (~30%)

In [ ]:
assert os.path.exists(ZS_CHECKPOINT), '❌ Chạy cell Download checkpoint trước'
!python test.py \
    --work_dir 'workdir/zeroshot_results' \
    --data_path 'dataset_BTXRD' \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode shift \
    --sam_checkpoint {ZS_CHECKPOINT} \
    --save_pred True


### C – Mixed 70/30

In [ ]:
assert os.path.exists(ZS_CHECKPOINT), '❌ Chạy cell Download checkpoint trước'
!python test.py \
    --work_dir 'workdir/zeroshot_results' \
    --data_path 'dataset_BTXRD' \
    --batch_size 1 \
    --image_size 256 \
    --boxes_prompt True \
    --prompt_mode mixed_7_3 \
    --sam_checkpoint {ZS_CHECKPOINT} \
    --save_pred True


### Kết quả & So sánh với Fine-tuned + PGA-UNet

In [ ]:
# ── Kết quả image-level từ prediction PNGs của SAM-Med2D ─────────────────────
# test.py lưu prediction per-polygon PNG vào PRED_DIRS.
# Chúng ta gom theo ảnh gốc, max-merge, tính metric image-level.
import cv2, numpy as np, os, csv
from scipy.ndimage import binary_erosion, distance_transform_edt

MODES    = ['zoom_out','shift','mixed_7_3']
PRED_DIRS = {m: f"workdir/zeroshot_results/sammed/boxes_{m}" for m in MODES}
MASK_DIR  = "dataset_BTXRD/test/masks"   # per-polygon GT PNGs: IMG001768_1.png
IMG_DIR_Z = "dataset_BTXRD/test/images"

S = 256   # SAM image size

def calc_metrics_img_zs(prob, gt, eps=1e-6):
    pm=(prob>0.5).astype(np.float32); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(S)
    if p.any() and g.any():
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS = ['dice','iou','precision','recall','hd95','cbl']
img_results = {}
zs_results = {}

for mode in MODES:
    pred_dir = PRED_DIRS[mode]
    if not os.path.exists(pred_dir):
        print(f'  ⚠ {mode}: không tìm thấy {pred_dir} — chạy test.py trước')
        img_results[mode] = {}; continue
    # Gom per-polygon GT và prediction theo ảnh gốc (stem)
    stem_data = {}
    for fn in sorted(os.listdir(MASK_DIR)):
        if not fn.endswith('.png'): continue
        parts = fn.rsplit('_', 1)
        if len(parts) < 2: continue
        stem = parts[0]
        gt_png  = cv2.imread(os.path.join(MASK_DIR, fn), cv2.IMREAD_GRAYSCALE)
        pred_png = cv2.imread(os.path.join(pred_dir, fn), cv2.IMREAD_GRAYSCALE)
        if gt_png is None or pred_png is None: continue
        gt256   = cv2.resize(gt_png,  (S,S), interpolation=cv2.INTER_NEAREST).astype(np.float32)/255.
        pred256 = cv2.resize(pred_png,(S,S), interpolation=cv2.INTER_NEAREST).astype(np.float32)/255.
        if stem not in stem_data:
            stem_data[stem] = dict(gt_union=gt256.copy(), prob_max=pred256.copy())
        else:
            np.maximum(stem_data[stem]['gt_union'],  gt256,   out=stem_data[stem]['gt_union'])
            np.maximum(stem_data[stem]['prob_max'],  pred256, out=stem_data[stem]['prob_max'])
    recs = {}
    for stem, d in sorted(stem_data.items()):
        recs[stem] = calc_metrics_img_zs(d['prob_max'], d['gt_union'])
    img_results[mode] = recs
    m_avg = {k: np.mean([r[k] for r in recs.values()]) for k in KEYS}
    zs_results[mode] = m_avg
    print(f"  [{mode:12s}] N={len(recs)} imgs | Dice={m_avg['dice']:.4f}  IoU={m_avg['iou']:.4f}"
          f"  HD95={m_avg['hd95']:.2f}px  CBL={m_avg['cbl']:.4f}")

os.makedirs('results', exist_ok=True)
with open('results/zeroshot_image_level.csv','w',newline='') as f:
    import csv as _csv
    w = _csv.writer(f); w.writerow(['mode']+KEYS+['N_img'])
    for mode in MODES:
        if not img_results.get(mode): continue
        recs = img_results[mode]
        m = {k: np.mean([r[k] for r in recs.values()]) for k in KEYS}
        w.writerow([mode]+[f'{m[k]:.4f}' for k in KEYS]+[str(len(recs))])
print('✅ CSV: results/zeroshot_image_level.csv')


In [ ]:
# ── Visualization: ≥10 ảnh có từ 2 GT polygon trở lên (zoom_out, image-level merged) ─
import matplotlib.pyplot as plt
import numpy as np
import cv2, os, json, glob as _glob
from matplotlib.patches import Patch, Rectangle
from IPython.display import display as _ipy_display

PRED_DIR    = "workdir/zeroshot_results/sammed/boxes_zoom_out"
MASK_DIR    = "dataset_BTXRD/test/masks"
S           = 256
N_MIN       = 10
MIN_GT      = 2
MODEL_LABEL = "SAM-Med2D Zero-Shot"

assert os.path.exists(PRED_DIR), f'❌ Chưa có prediction — chạy cell Test (zoom_out) trước\n  {PRED_DIR}'

# ── 1. Gom per-polygon → image-level (max-merge) ────────────────────────────
with open("dataset_BTXRD/label2image_test.json") as f:
    label2image = json.load(f)

stem_data = {}
for mask_rel, img_rel in label2image.items():
    mask_name = os.path.basename(mask_rel)
    parts = mask_name.rsplit('_', 1)
    if len(parts) < 2: continue
    stem = parts[0]

    gt_p = cv2.imread(mask_rel, cv2.IMREAD_GRAYSCALE)
    pr_p = cv2.imread(os.path.join(PRED_DIR, mask_name), cv2.IMREAD_GRAYSCALE)
    if gt_p is None: continue
    gt256 = cv2.resize(gt_p, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    if gt256.max() > 1: gt256 /= 255.
    pr256 = np.zeros((S, S), dtype=np.float32)
    if pr_p is not None:
        pr256 = cv2.resize(pr_p, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.float32)
        if pr256.max() > 1: pr256 /= 255.

    if stem not in stem_data:
        img_bgr = cv2.imread(img_rel, cv2.IMREAD_GRAYSCALE)
        img256  = cv2.resize(img_bgr, (S, S)) if img_bgr is not None else np.zeros((S, S), np.uint8)
        stem_data[stem] = dict(gt_union=gt256.copy(), prob_max=pr256.copy(),
                               n_poly=1, img=img256.astype(np.float32) / 255.)
    else:
        np.maximum(stem_data[stem]['gt_union'], gt256, out=stem_data[stem]['gt_union'])
        np.maximum(stem_data[stem]['prob_max'], pr256, out=stem_data[stem]['prob_max'])
        stem_data[stem]['n_poly'] += 1

# ── 2. Tính metrics & filter ≥MIN_GT polygon ────────────────────────────────
eps = 1e-6
def _calc(prob, gt):
    pm = (prob > 0.5).astype(np.float32); gm = (gt > 0.5).astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),
                iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),
                recall=float((tp+eps)/(tp+fn+eps)))

recs = []
for stem in sorted(stem_data.keys()):
    d = stem_data[stem]
    if d['n_poly'] < MIN_GT: continue
    m = _calc(d['prob_max'], d['gt_union'])
    recs.append(dict(img_name=stem, img=d['img'],
                     gt=d['gt_union'], prob=d['prob_max'],
                     n_poly=d['n_poly'], **m))

assert len(recs) >= N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT (cần {N_MIN})'
recs = recs[:max(N_MIN, len(recs))]
N_SHOW = len(recs)

# ── 3. Vẽ N_SHOW × 5 (giống PGA) ────────────────────────────────────────────
fig, axes = plt.subplots(N_SHOW, 5, figsize=(20, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]
fig.suptitle(f'{MODEL_LABEL} — {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level merged)',
             fontsize=13, y=1.001)

col_titles = ['Ảnh gốc', 'Ảnh + Bbox Prompts', 'Dự đoán (merged)', 'Ground Truth', 'TP/FP/FN']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10, fontweight='bold')

def get_per_poly_bboxes(stem, r=0.30):
    """Đọc từng mask PNG per-polygon → tính bbox zoom-out riêng từng cái."""
    bboxes = []
    for pm_path in sorted(_glob.glob(os.path.join(MASK_DIR, f"{stem}_*.png"))):
        pm_raw = cv2.imread(pm_path, cv2.IMREAD_GRAYSCALE)
        if pm_raw is None: continue
        pm256 = cv2.resize(pm_raw, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.float32)
        if pm256.max() > 1: pm256 /= 255.
        ys_p, xs_p = np.where(pm256 > 0.5)
        if len(xs_p) == 0: continue
        x0, x1 = int(xs_p.min()), int(xs_p.max())
        y0, y1 = int(ys_p.min()), int(ys_p.max())
        gw, gh = x1 - x0, y1 - y0
        bx0 = max(0, x0 - gw * r); bx1 = min(S, x1 + gw * r)
        by0 = max(0, y0 - gh * r); by1 = min(S, y1 + gh * r)
        bboxes.append((bx0, by0, bx1 - bx0, by1 - by0))  # x, y, w, h
    return bboxes

for count, rec in enumerate(recs):
    img_np  = rec['img']
    gt_np   = (rec['gt']   > 0.5).astype(float)
    pred_np = (rec['prob'] > 0.5).astype(float)
    n       = rec['n_poly']
    dice    = rec['dice'];  iou  = rec['iou']
    pre     = rec['precision']; rec_ = rec['recall']

    row = axes[count]
    bg  = np.stack([img_np] * 3, axis=-1)

    # Col 0 – Ảnh gốc
    row[0].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    row[0].set_ylabel(f'#{count+1} [{n} poly]\nDice={dice:.3f}', fontsize=8)

    # Col 1 – Ảnh + per-polygon bbox riêng lẻ (như PGA hiện nhiều hotspot riêng)
    row[1].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    for bbox in get_per_poly_bboxes(rec['img_name']):
        row[1].add_patch(Rectangle((bbox[0], bbox[1]), bbox[2], bbox[3],
                                   linewidth=2, edgecolor='yellow', facecolor='none'))
    row[1].set_title(rec['img_name'], fontsize=6, pad=2)

    # Col 2 – Dự đoán (red overlay)
    pr_ov = bg.copy()
    pr_ov[..., 0] = np.clip(pr_ov[..., 0] + pred_np * 0.55, 0, 1)
    pr_ov[..., 1] = np.clip(pr_ov[..., 1] - pred_np * 0.2,  0, 1)
    row[2].imshow(pr_ov)
    row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}', fontsize=7, color='darkred', pad=2)

    # Col 3 – Ground Truth (green overlay)
    gt_ov = bg.copy()
    gt_ov[..., 1] = np.clip(gt_ov[..., 1] + gt_np * 0.55, 0, 1)
    gt_ov[..., 0] = np.clip(gt_ov[..., 0] - gt_np * 0.2,  0, 1)
    row[3].imshow(gt_ov)

    # Col 4 – TP/FP/FN (TP=xanh lá, FP=đỏ, FN=xanh dương)
    inter = bg.copy() * 0.25
    inter[..., 1] = np.clip(inter[..., 1] + pred_np * gt_np       * 0.9, 0, 1)
    inter[..., 0] = np.clip(inter[..., 0] + pred_np * (1 - gt_np) * 1.0, 0, 1)
    inter[..., 2] = np.clip(inter[..., 2] + (1 - pred_np) * gt_np * 1.0, 0, 1)
    row[4].imshow(inter)
    row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}', fontsize=7, color='saddlebrown', pad=2)

    for ax in row:
        ax.axis('off')

fig.legend(
    handles=[Patch(facecolor='green', label='TP'),
             Patch(facecolor='red',   label='FP'),
             Patch(facecolor='blue',  label='FN')],
    loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.005)
)
plt.tight_layout()
_ipy_display(fig)
plt.close(fig)

---
# PHẦN 4 — Tổng hợp & So sánh 3 mô hình
---

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TỔNG HỢP SO SÁNH (zoom_out): PGA-UNet vs SAM-Med2D Zero-Shot vs SAM-Med2D Fine-tuned
# Lấy TRỰC TIẾP từ kết quả đã tính ở các cell Test phía trên — KHÔNG hard-code số liệu
# ══════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

assert 'pga_results' in dir(), '❌ Chưa chạy cell Test PGA-UNet (3 prompt modes) ở trên!'
assert 'zs_results'  in dir(), '❌ Chưa chạy cell kết quả SAM-Med2D Zero-Shot ở trên!'
assert 'ft_results'  in dir(), '❌ Chưa chạy cell kết quả SAM-Med2D Fine-tuned ở trên!'

MODE = 'zoom_out'
for _name, _res in [('PGA-UNet', pga_results), ('SAM-Med2D Zero-Shot', zs_results),
                     ('SAM-Med2D Fine-tuned', ft_results)]:
    assert MODE in _res, f'❌ {_name}: chưa có kết quả mode={MODE!r}, chạy lại cell Test tương ứng!'

data_zo = {
    'PGA-UNet':             pga_results[MODE],
    'SAM-Med2D Zero-Shot':  zs_results[MODE],
    'SAM-Med2D Fine-tuned': ft_results[MODE],
}

MODELS  = list(data_zo.keys())
KEYS    = ['dice', 'iou', 'precision', 'recall', 'hd95', 'cbl']
HDRS    = ['Dice\u2191', 'IoU\u2191', 'Prec\u2191', 'Rec\u2191', 'HD95\u2193 (px)', 'CBL\u2191']
METRICS = list(zip(KEYS, ['Dice \u2191', 'IoU \u2191', 'Precision \u2191',
                           'Recall \u2191', 'HD95 \u2193 (px)', 'CBL \u2191']))
COLORS  = ['#2196F3', '#F44336', '#4CAF50']   # PGA=blue, ZS=red, FT=green
X_LBLS  = ['PGA-UNet', 'SAM-ZS', 'SAM-FT']

# ── 1. Bảng tổng hợp ──────────────────────────────────────────────────────────
bar = '\u2550' * 80
print(f'\n{bar}')
print('  TỔNG HỢP SO SÁNH (zoom_out) — Image-level metrics (N\u224887 ảnh test)')
print(f'{bar}')
print(f"  {'Model':<26}" + ''.join(f'{h:>10}' for h in HDRS))
print('  ' + '-' * 77)
for model, m in data_zo.items():
    print(f"  {model:<26}" + ''.join(f'{m[k]:>10.4f}' for k in KEYS))
print(f'{bar}')
for k in KEYS:
    suffix = '  (âm = PGA tốt hơn)' if k == 'hd95' else ''
    for comp in ['SAM-Med2D Zero-Shot', 'SAM-Med2D Fine-tuned']:
        delta = data_zo['PGA-UNet'][k] - data_zo[comp][k]
        tag   = 'ZS' if 'Zero' in comp else 'FT'
        print(f'  \u0394 PGA-UNet \u2212 SAM-{tag}  [{k.upper():<12}]: {delta:+.4f}{suffix}')
    print()

# ── 2. Biểu đồ cột ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
fig.suptitle('So sánh 3 mô hình (zoom_out): PGA-UNet | SAM-Med2D Zero-Shot | SAM-Med2D Fine-tuned\n'
             '(Image-level, GT union + max-merge, N\u224887 ảnh test, IMG_SIZE=256)',
             fontsize=13, fontweight='bold', y=1.01)

x = np.arange(len(MODELS))
for ax_idx, (metric_key, metric_label) in enumerate(METRICS):
    ax   = axes[ax_idx]
    vals = [data_zo[m][metric_key] for m in MODELS]

    bars = ax.bar(x, vals, 0.5, color=COLORS, alpha=0.88, edgecolor='white')
    for bar_, v in zip(bars, vals):
        fmt = f'{v:.0f}' if metric_key == 'hd95' else f'{v:.4f}'
        ax.text(bar_.get_x() + bar_.get_width()/2,
                bar_.get_height() + (0.004 if metric_key != 'hd95' else 0.8),
                fmt, ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(metric_label, fontsize=12, fontweight='bold', pad=8)
    hi = max(vals)
    ax.set_ylim(0, hi * 1.22 if metric_key != 'hd95' else hi * 1.30)
    ax.set_xticks(x); ax.set_xticklabels(X_LBLS, fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

handles = [mpatches.Patch(color=c, alpha=0.88, label=l)
           for c, l in zip(COLORS, ['PGA-UNet', 'SAM-Med2D Zero-Shot', 'SAM-Med2D Fine-tuned'])]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('/kaggle/working/comparison_3models_zoomout.png', dpi=150, bbox_inches='tight')
from IPython.display import display as _ipy_display
_ipy_display(fig)
plt.close(fig)
print('\n✅ Biểu đồ đã lưu: comparison_3models_zoomout.png')